# ChartQA rollout generation (cross-dataset transfer test)

Generates 5 rollouts per question for the 150-question ChartQA train pool (human-authored, non-yes/no test-split questions), using the exact same prompt and generation settings as the original CharXiv pipeline (`chart_prm.generator.build_generation_prompt`), so the only variable that changes is the source dataset.

Self-contained: reads the pool from the `chartqa-transfer-pool` Kaggle Dataset (not a git clone), and inlines the generation prompt directly rather than importing from the repo, so this kernel has no GitHub dependency at all.


In [ ]:
# v1-v4 root causes fixed (see prior comments in kernel history): unpinned
# bitsandbytes CUDA mismatch on the T4 image; torch/transformers version skew;
# in-process reinstall not taking effect because torch was imported too early.
# v5's new error: on THIS P100 image variant, bitsandbytes isn't installed at
# all (PackageNotFoundError) -- apparently the P100 base image doesn't bundle it
# the way the T4 image does (that's a different problem than v1's, where it WAS
# present and an unpinned reinstall broke it). Fix: install it explicitly, but
# only in the P100 branch, pinned to a version validated against torch 2.5.1+cu124.
import subprocess, sys

gpu_names = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip().splitlines()
print('Detected GPUs (nvidia-smi):', gpu_names)
is_p100 = any('P100' in n for n in gpu_names)

if is_p100:
    print('*** Tesla P100 detected. Installing the validated older stack (torch 2.5.1+cu124, transformers 4.49.0, bitsandbytes 0.45.0)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.49.0', 'accelerate==1.2.1', 'bitsandbytes==0.45.0'], check=True)
else:
    print('Non-P100 GPU: upgrading transformers to a current release (floor-version style, no upper pin). Not touching bitsandbytes -- the T4 image already ships a working one (v1 broke it by reinstalling unpinned).')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.49.0', 'accelerate'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'qwen-vl-utils==0.0.14'], check=True)

# First import of torch/transformers in this process happens here, after every
# install above -- guaranteed to load what's actually on disk now.
import torch
import transformers
print(f'Active PyTorch: {torch.__version__}, transformers: {transformers.__version__}, CUDA available: {torch.cuda.is_available()}')


In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config
)

processor = AutoProcessor.from_pretrained(model_id)


In [ ]:
import glob
import json
import os
from PIL import Image

# Don't assume the mount path -- v6 guessed /kaggle/input/chartqa-transfer-pool
# and got FileNotFoundError even though the dataset's own file listing confirmed
# chartqa_reasoning.json is there. Locate it by search instead of guessing again.
print("Contents of /kaggle/input:", os.listdir("/kaggle/input"))
matches = glob.glob("/kaggle/input/**/chartqa_reasoning.json", recursive=True)
print("Found chartqa_reasoning.json at:", matches)
assert matches, "chartqa_reasoning.json not found anywhere under /kaggle/input"
INPUT_DIR = os.path.dirname(matches[0])
print("Using INPUT_DIR =", INPUT_DIR)

with open(os.path.join(INPUT_DIR, "chartqa_reasoning.json"), "r", encoding="utf-8") as f:
    all_questions = json.load(f)

with open(os.path.join(INPUT_DIR, "chartqa_main_ids.json"), "r", encoding="utf-8") as f:
    train_ids = json.load(f)

print(f"Train-pool questions: {len(train_ids)}")
assert len(train_ids) == 150


In [ ]:
def build_generation_prompt(question: str, extra_instruction: str = "") -> str:
    """
    Identical to chart_prm.generator.build_generation_prompt (CharXiv pipeline) --
    inlined here so this kernel has no GitHub dependency. Keep in sync with
    src/chart_prm/generator.py if that prompt ever changes.
    """
    prompt = (
        "You are an expert at extracting data and reasoning about scientific charts. "
        "Carefully inspect the axes, axis labels, legend, colors, markers, values, units, titles, ticks, and trends. "
        f"{extra_instruction} "
        "If the question is not related to the image or there is not enough information to answer, output 'Not Applicable'.\n\n"
        "RULES FOR REASONING:\n"
        "1. DO NOT write any introductory or conversational text. Begin immediately with 'Step 1:'.\n"
        "2. Break down your thought process into explicit, logical reasoning steps.\n"
        "3. DO NOT output a high-level plan (e.g., 'Find the x-axis'). You MUST explicitly state the concrete values, labels, and colors you read from the chart in each step.\n"
        "4. Perform and display explicit comparisons and intermediate math calculations.\n"
        "5. Label each step on a new line starting exactly with 'Step 1:', 'Step 2:', etc.\n"
        "6. Provide the final concise answer on a new line starting strictly with 'Final Answer:'. The final answer MUST be ONLY the exact short value or entity.\n\n"
        "---\n"
        "EXAMPLE FORMAT:\n"
        "Question: Which model has the highest accuracy at Epoch 10?\n"
        "Step 1: The x-axis represents 'Epochs'. I need to find the data points at the vertical line for Epoch 10.\n"
        "Step 2: At Epoch 10, Model A (blue line) has an accuracy of approximately 72%.\n"
        "Step 3: At Epoch 10, Model B (red line) has an accuracy of approximately 85%.\n"
        "Step 4: At Epoch 10, Model C (green line) has an accuracy of approximately 60%.\n"
        "Step 5: Comparing the extracted values: 85% > 72% > 60%.\n"
        "Final Answer: Model B\n"
        "---\n\n"
        f"Question: {question}"
    )
    return prompt


In [ ]:
output_file = "generated_chartqa_rollouts.jsonl"
NUM_ROLLOUTS = 5
save_every = 5

start_index = 0
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)
        start_index = total_lines // NUM_ROLLOUTS
    print(f"Found existing checkpoint with {total_lines} trajectories. Resuming from sample {start_index}")

num_samples = len(train_ids)

for i, qid in enumerate(train_ids):
    if i < start_index:
        continue

    q = all_questions[qid]
    question = q["query"]
    ground_truth = q["answer"]
    image_path = os.path.join(INPUT_DIR, "images", f"{qid}.jpg")
    image = Image.open(image_path).convert("RGB")

    prompt_text = build_generation_prompt(question)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    # Generate sequentially to avoid VRAM OOM, same as the original pipeline
    for rollout_idx in range(NUM_ROLLOUTS):
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.7
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        res = {
            "question_id": qid,
            "rollout_index": rollout_idx,
            "question": question,
            "model_output": output_text,
            "ground_truth": ground_truth,
        }

        with open(output_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(res, ensure_ascii=False) + "\n")

    if (i + 1) % save_every == 0 or (i + 1) == num_samples:
        print(f"Processed {i+1}/{num_samples} questions ({NUM_ROLLOUTS} rollouts each)...")

print(f"Finished generating {num_samples * NUM_ROLLOUTS} trajectories!")
